In [0]:
%sql
create connection if not exists dlt_pipeline_conn
type HTTP
options (
  host = 'https://earthquake.usgs.gov',
  port = 443,
  base_path = '/earthquakes/feed/v1.0/',
  bearer_token = 'na'
)

In [0]:
from databricks.sdk import WorkspaceClient
wc = WorkspaceClient()

conn = wc.connections.get("dlt_pipeline_conn")
base_url = f"{conn.options['host']}{conn.options['base_path']}"

In [0]:
%sql
use catalog dlt_pipeline_project;
use schema bronze;

create volume if not exists earthquake_data;

In [0]:
import requests
import json

url = f"{base_url}summary/all_day.geojson"
response = requests.get(url)

if response.status_code!=200:
  raise Exception(f"Error in getting data from {url}")
else:
  data = response.json()
  dbutils.fs.put('/Volumes/dlt_pipeline_project/bronze/earthquake_data/earthquake_data.json',json.dumps(data), overwrite=True)